# Cost per asset

Run this notebook on your **laptop**. It turns one run's measurements into a cost per asset, at the regular price and at today's preemptible price, and compares it with a per-call API.

No GPU needed: use our sample run, or the `metrics.json` from your own Job (lesson 04) or Endpoint run.

You're done when you have the cost table, the preemptible difference, and a breakeven against one API price with its link.

Guide: [05-measuring-costs.md](../05-measuring-costs.md).

In [ ]:
import sys
from pathlib import Path

MODULE = "01-consistent-media-generation-on-serverless-gpus"
candidates = [Path.cwd(), Path.cwd() / "code", Path.cwd() / MODULE / "code"]
found = [p for p in candidates if (p / "common.py").is_file()]
assert found, "Open this notebook from the repository (its code/ folder)"
CODE = found[0].resolve()
sys.path.insert(0, str(CODE))

## 1. Load one run's measurements

- `startup_s`: create → `RUNNING`.
- `t_ready`: Endpoint: `RUNNING` → first answer (`ready_wait_s`). Job: model load (`load_s`).
- `latency_s`: generation time per asset; we use the median.

In [ ]:
import json
import statistics

# [placeholder: sample run from the author run] Or point this at your own metrics.json.
METRICS = CODE / "sample_runs" / "sana-l40s-job" / "metrics.json"

m = json.loads(METRICS.read_text())
t_start = m["startup_s"]
t_ready = m["ready_wait_s"] if m["mode"] == "endpoint" else m["load_s"]
latency = m["latency_s"]
print(f"{m['mode']} on {m['env'].get('gpu')}: startup {t_start} s, ready/load {t_ready} s, "
      f"median generation {statistics.median(latency):.1f} s over {len(latency)} assets")

## 2. Regular price (the default)

Open [Compute pricing](https://docs.nebius.com/compute/resources/pricing) and find the run's GPU. Type the regular (on-demand) price per hour for the preset you ran.

L40S lists GPU, vCPU and RAM separately: for `1gpu-8vcpu-32gb` add GPU + 8 × vCPU + 32 × RAM. H100 and H200 list one price per GPU hour.

In [ ]:
REGULAR_RATE = None  # USD per hour for your preset, typed from the pricing page
PRICE_DATE = "YYYY-MM-DD"
assert REGULAR_RATE, "Type the regular rate from the pricing page"

## 3. Cost per asset

- Marginal cost per asset: `c = rate × t_gen`.
- Fixed cost per run: `F = rate × (t_start + t_ready)`. Idle time adds `rate × t_idle`.
- Total per asset for N assets: `F / N + c`.

Assumptions: one startup per run, assets generated one after another, no idle time unless added. Left out: failed or retried generations, engineering and ops time, storage and egress, and quality differences against an API. Each one makes self-hosting look cheaper than it is.

In [ ]:
from common import calculate_cost


def show(rate, label, idle_s=0, api_price=None):
    r = calculate_cost(t_start, t_ready, latency, rate, idle_s=idle_s, api_price=api_price)
    per_asset = ", ".join(f"N={n}: ${v:.4f}" for n, v in r["projected_compute_usd_per_asset"].items())
    print(f"{label:<24} c = ${r['marginal_compute_usd']:.5f} (marginal), F = ${r['fixed_run_usd']:.4f} | {per_asset}")
    return r


show(REGULAR_RATE, "regular")
show(REGULAR_RATE, "regular + 10 min idle", idle_s=600)

## 4. Today's preemptible price

The course runs on preemptible VMs. For GPUs with a spot price (H100, H200, B200, B300, RTX PRO 6000) the price moves during the day; `spot_price` reads it from the public API. L40S isn't listed: its preemptible price is fixed on the pricing page.

In [ ]:
from datetime import datetime, timezone

from common import spot_price

PLATFORM = "gpu-l40s-a"  # the platform your run used
FIXED_PREEMPTIBLE_RATE = None  # L40S only: the fixed preemptible price for your preset (GPU + vCPU + RAM), from the pricing page
try:
    SPOT_RATE, region, since = spot_price(PLATFORM)
    print(f"{PLATFORM}: ${SPOT_RATE}/GPU-hour in {region}, since {since} (read {datetime.now(timezone.utc):%H:%M} UTC)")
except ValueError as error:
    print(error)
    SPOT_RATE = FIXED_PREEMPTIBLE_RATE
assert SPOT_RATE, "No spot price for this platform: type FIXED_PREEMPTIBLE_RATE from the pricing page"

show(REGULAR_RATE, "regular")
show(SPOT_RATE, "preemptible today")

## 5. Breakeven against a per-call API

Find a service that sells the same model (or the closest one) per call. Write down its price per asset for your settings and the link.

Breakeven N = assets per run where self-hosting gets cheaper than the API. `0` means the API is cheaper per asset than your marginal cost, at any N.

In [ ]:
API_PRICE = None  # USD per asset, from the provider's pricing page
API_LINK = ""
assert API_PRICE and API_LINK, "Type the API price and its link"

for rate, label in ((REGULAR_RATE, "regular"), (SPOT_RATE, "preemptible today")):
    if rate:
        print(label, "breakeven N =", show(rate, label, api_price=API_PRICE)["breakeven_assets"])

## 6. Check what you spent

The console shows your balance under **Billing** and the spend per project, service and resource under **Billing → Usage**. The docs describe no separate "credits remaining" view. Compare the usage for your run with `billable_s × rate`.

You're done when:

- [ ] You have the cost table at the regular price, with and without idle time.
- [ ] You have the preemptible difference for today.
- [ ] You have a breakeven N against one API price, with its link.